# This notebook is to sort by channel and then cluster each channel

**USE Cleaned Data!**

In [1]:
from treeview_gen import df_to_treeview
import pandas as pd
import os
import re
import shutil
from time import sleep

regex_df = pd.read_csv("/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned/HistDiff_results_24h_bmal1ref_FeatRed_WELL-DROPPED.csv", index_col=0, nrows=1)
pattern = r'Region (.+?) \(global\)'
channels_found = set()
for col in regex_df.columns:
    found = re.search(pattern, col)
    if found:
        channels_found.add(found.group(1))

print(channels_found)


{'MitoTracker Deep Red', 'Alexa 488 shortened', 'SYTO14 Rabbitts', 'HOECHST 33342 expanded', 'TMR/AF555 Rabbitts'}


In [2]:
def gen_treeview_channel(df: pd.DataFrame, channel_name: str, out_dir: str) -> list[str]:
    """Generates a treeviewe file set for a given channel

    Args:
        df (pd.DataFrame): channel data frame
        channel_name (str): the name of the channel
        out_dir (str): the file path where the file will be outputted to (this creates a folder)
    """
    
    out_dir_channel = os.path.join(out_dir, channel_name)
    
    if os.path.exists(out_dir_channel):
        shutil.rmtree(out_dir_channel)
        sleep(0.5)
    os.makedirs(out_dir_channel, exist_ok=True)
    
    out_dir_channel = os.path.join(out_dir_channel, f"{channel_name.replace("/","_")}_treeview")
    
    df_to_treeview(df=df, outFile=out_dir_channel, sortIndex=True, row_cluster=False)
    
    cdt_path = f"{out_dir_channel}.cdt"
    return read_tv_col_order(cdt_path)


def read_tv_col_order(file_path: str) -> list[str]:
    col_df = pd.read_csv(file_path, index_col=[0,1,2], sep="\t")
    return col_df.columns.tolist()

In [52]:
test_cdt = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/HistDiff_results_24h_bmal1ref_FeatRed_WELL-DROPPED/Alexa 488 shortened/Alexa 488 shortened_treeview.cdt"
cdt_df = pd.read_csv(test_cdt, sep="\t", index_col=[0,1,2])

print(cdt_df.columns.tolist())

['Nuclei - Ring Region Alexa 488 shortened (global) Symmetry 02 SER-Saddle', 'Nuclei - Ring Region Alexa 488 shortened (global) Symmetry 02 SER-Edge', 'Nuclei - Ring Region Alexa 488 shortened (global) Symmetry 13 SER-Edge', 'Nuclei - Cytoplasm Region Alexa 488 shortened (global) Symmetry 12 SER-Saddle', 'Nuclei - Cytoplasm Region Alexa 488 shortened (global) Symmetry 02 SER-Saddle', 'Nuclei - Cytoplasm Region Alexa 488 shortened (global) Symmetry 15 SER-Saddle', 'Nuclei - Cytoplasm Region Alexa 488 shortened (global) Symmetry 05 SER-Saddle', 'Nuclei - Membrane Region Alexa 488 shortened (global) Symmetry 05 SER-Saddle', 'Nuclei - Cell Region Alexa 488 shortened (global) Symmetry 05 SER-Saddle', 'Nuclei - Cell Region Alexa 488 shortened (global) Symmetry 15 SER-Saddle', 'Nuclei - Cytoplasm Region Alexa 488 shortened (global) Symmetry 03 SER-Saddle', 'Nuclei - Cytoplasm Region Alexa 488 shortened (global) Symmetry 13 SER-Saddle', 'Nuclei - Cell Region Alexa 488 shortened (global) Symmet

## Final Loop

In [ ]:
cleaned_file_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned"
out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted"

for file in os.listdir(cleaned_file_dir):
    curr_file = os.path.join(cleaned_file_dir, file)
    print(curr_file)
    raw_name = os.path.splitext(file)[0]
    print(raw_name)
    
    raw_out_dir = os.path.join(out_dir, raw_name)
    print(raw_out_dir)
    
    # make a folder
    if os.path.exists(raw_out_dir):
        shutil.rmtree(raw_out_dir)
        sleep(0.5)
    os.makedirs(raw_out_dir)
    
    
    curr_df = pd.read_csv(curr_file, index_col=0)
    
    # grab channels from data
    channel_df = {ch: curr_df[[col for col in curr_df.columns if ch in col]] for ch in channels_found}
    
    # grab the column orders and export out the channel clustering
    col_order = []
    for name, df in channel_df.items():
        order = gen_treeview_channel(df, name.replace("/", "_"), raw_out_dir)
        col_order.extend(order)
    
    print(col_order)
    
    reorder_df = curr_df[col_order]
    
    final_out_path = os.path.join(raw_out_dir, f"{raw_name}_SORTED-CLUSTERED_channels")
    df_to_treeview(df=reorder_df, outFile=final_out_path, sortIndex=True, row_cluster=False, col_cluster=False)

/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned/HistDiff_results_24h_bmal1ref_FeatRed_WELL-DROPPED.csv
HistDiff_results_24h_bmal1ref_FeatRed_WELL-DROPPED
/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/HistDiff_results_24h_bmal1ref_FeatRed_WELL-DROPPED
['Nuclei - Membrane Region SYTO14 Rabbitts (global) Symmetry 15 SER-Spot', 'Nuclei - Membrane Region SYTO14 Rabbitts (global) Symmetry 13 SER-Ridge', 'Nuclei - Ring Region SYTO14 Rabbitts (global) Symmetry 03 SER-Saddle', 'Nuclei - Ring Region SYTO14 Rabbitts (global) Symmetry 05 SER-Saddle', 'Nuclei - Membrane Region SYTO14 Rabbitts (global) Symmetry 04 SER-Edge', 'Nuclei - Membrane Region SYTO14 Rabbitts (global) Symmetry 04 SER-Saddle', 'Nuclei - Membrane Region SYTO14 Rabbitts (global) Symmetry 12 SER-Hole', 'Nuclei - Cytoplasm Region SYTO14 Rabbitts (global) Symmetry 12 SER-Hole', 'Nuclei - Cytoplasm Region SYTO14 Rabbitts (global) Symmet

# We made an error:

- **ALL HEATMAPS MUST HAVE THE SAME COLUMN ORDERINGS AND CLUSTERING**
- This means we need to do the following:
1. Combine all the datasets together
2. Split channels
3. cluster columns of combined but channel split dataset
4. retrieve ordering
5. sort ids and apply ordering

In [4]:
cleaned_file_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned"
out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted"

In [3]:
# lets remove the orginal channel sorted folder
def clean_folder(path: str) -> None:
    if os.path.exists(path):
        shutil.rmtree(path)
        sleep(0.5)
    os.makedirs(path)

In [5]:
clean_folder(out_dir)

In [14]:
# combine data sets
renamed_datasets = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned_renamed"

dfs = []
for file in os.listdir(renamed_datasets):
    curr_file = os.path.join(renamed_datasets, file)
    curr_df = pd.read_csv(curr_file, index_col=0)
    dfs.append(curr_df)

concat_df = pd.concat(dfs)
display(concat_df)
print(concat_df.shape)

concat_df.to_csv('/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/combined_data.csv')

,Nuclei - Cell Region Alexa 488 shortened (global) Profile 2/5 SER-Hole,Nuclei - Ring Region HOECHST 33342 expanded (global) Radial Relative Deviation SER-Ridge,Nuclei - Ring Region SYTO14 Rabbitts (global) Profile 2/5 SER-Hole,Nuclei - Cell Region TMR/AF555 Rabbitts (global) Profile 3/5 SER-Bright,Nuclei - Nucleus Region SYTO14 Rabbitts (global) Radial Mean Ratio SER-Valley,Nuclei - Intensity Membrane Region MitoTracker Deep Red (global) Contrast,Nuclei - Cytoplasm Region MitoTracker Deep Red (global) Threshold Compactness 30%,Nuclei - Cell Region MitoTracker Deep Red (global) Profile 2/5,Nuclei - Nucleus Region HOECHST 33342 expanded (global) Symmetry 13 SER-Bright,Nuclei - Ring Region Alexa 488 shortened (global) Profile 1/5,...,Nuclei - Nucleus Region HOECHST 33342 expanded (global) Symmetry 05 SER-Bright,Nuclei - Membrane Region HOECHST 33342 expanded (global) Profile 1/5,Nuclei - Cytoplasm Region MitoTracker Deep Red (global) Threshold Compactness 40%,Nuclei - Membrane Region MitoTracker Deep Red (global) Threshold Compactness 30% SER-Edge,Nuclei - Ring Region HOECHST 33342 expanded (global) Gabor Min 2 px w2,Nuclei - Ring Region MitoTracker Deep Red (global) Radial Mean Ratio SER-Edge,Nuclei - Nucleus Region TMR/AF555 Rabbitts (global) Threshold Compactness 30%,Nuclei - Cell Region HOECHST 33342 expanded (global) Threshold Compactness 30%,Nuclei - Intensity Ring Region HOECHST 33342 expanded (global) StdDev,Nuclei - Membrane Region HOECHST 33342 expanded (global) Profile 2/5 SER-Hole
human_readable_ids,,,,,,,,,,,,,,,,,,,,,
NO-TREATMENT._.A10._.24h,0.002570,-0.000127,0.000008,-0.000064,0.000220,-0.000272,-0.000084,-0.000118,-0.000523,0.000083,...,-0.000169,0.000184,0.000054,-0.000040,1.205608e-04,-0.000057,-0.000042,-0.000877,0.003708,-1.190946e-05
NO-TREATMENT._.A11._.24h,0.000060,0.000244,-0.000007,-0.000014,-0.000093,-0.001290,0.000673,0.000098,-0.000067,-0.000018,...,-0.000095,0.000099,0.000093,-0.000161,-5.480439e-06,0.000100,0.000214,0.000330,0.002808,-8.364908e-06
NO-TREATMENT._.A12._.24h,-0.001167,-0.000180,-0.000007,-0.001610,-0.000058,0.001164,0.000586,0.000199,-0.000086,0.000102,...,-0.000194,-0.000259,-0.000050,-0.000040,-4.587930e-04,0.000036,0.000293,-0.000130,0.001278,-9.000905e-07
NO-TREATMENT._.A13._.24h,0.000782,0.000297,-0.000051,-0.000051,0.000504,-0.001473,-0.000314,-0.000191,0.000255,-0.000177,...,0.000504,0.000427,-0.000115,0.000018,-3.523404e-07,0.000231,-0.000153,0.001274,0.001728,2.906337e-06
NO-TREATMENT._.A14._.24h,-0.008598,-0.000412,0.000001,-0.000047,-0.000597,0.019484,-0.000103,0.001361,0.001008,0.002145,...,-0.000221,-0.000301,0.000267,0.001576,2.909721e-05,-0.001983,0.000151,-0.002635,0.001123,-9.838245e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PF_per2._.O05._.72h,-0.000008,0.000849,-0.000005,-0.001232,0.000042,0.019044,0.001375,0.000298,0.000576,0.000179,...,0.000281,-0.000183,0.001308,0.001528,-1.767191e-02,-0.001893,-0.000465,-0.008355,0.001306,1.373676e-06
PF_per2._.O06._.72h,-0.000020,0.001391,-0.000001,-0.004535,0.001633,0.005978,-0.000233,-0.000078,0.000216,0.000025,...,0.000272,0.000068,0.000205,0.001067,-3.699737e-03,-0.001330,-0.000802,-0.008293,0.000729,-1.046436e-06
PF_per2._.O07._.72h,0.000038,0.000798,-0.000023,-0.001648,0.000049,0.003919,0.000313,-0.000183,0.000989,0.000023,...,0.000189,0.000418,0.000600,0.000867,-2.904295e-02,-0.000786,-0.000855,-0.003459,0.000625,1.727853e-06


(1056, 879)


## Final loop